# Nonmonotone spread discovery

This study tests whether discovery can select a varying Gaussian noise scale whose signed slope is zero. The absolute-value scale process (a noise scale that depends on the absolute value of `x1`) receives a curve-sensitive nomination, a step that proposes a candidate scale term for the held-out score to test, followed by the existing held-out distributional comparison. The tables compare the slope-only baseline, the curve-sensitive nomination screen, a supplied-specification estimator, and five competing interval methods.

The headline results use 20 independent training/test draws per process. The known-specification track gives its estimator the mean formula and, on the varying-scale processes, a scale term for `x1`. The discovery track gives its estimator only the development data and the response name. The data-only competitors receive only the data in both tracks. Competitor settings are fixed and untuned. The comparison competitor for each cell is the one with the lowest width among those with eligible coverage. That choice uses the test results after the fact, so it is not a prespecified winner.

**Limitation.** Discovery selects on all development rows before its internal calibration subset is held out. The [calibration-boundary audit](nonmonotone_spread/BOUNDARY_AUDIT.md) records this. The results below describe empirical coverage, not a finite-sample guarantee for adaptive discovery. The nomination step alone does not remove the requirement that calibration rows be excluded from selection.


In [1]:
import json
from pathlib import Path
import numpy as np
import polars as pl
from great_tables import GT
from IPython.display import display

folder = Path("nonmonotone_spread")
baseline = json.loads((folder / "final_baseline.json").read_text())
fixed = json.loads((folder / "final_fixed.json").read_text())
records = baseline["results"] + fixed["results"]
assert len(baseline["results"]) == 700 and len(fixed["results"]) == 100
assert all(row["status"] == "ok" for row in records)
processes = ["absolute", "monotone", "bell", "constant", "heavy_tail"]

def average_se(values):
    values = np.asarray(values, dtype=float)
    return float(values.mean()), float(values.std(ddof=1) / np.sqrt(len(values)))


In [2]:
# Each uncertainty estimate is across independent data draws, not CV folds.
summary = []
for process in processes:
    for phase, source in [("main", baseline), ("curve", fixed)]:
        methods = list(dict.fromkeys(r["method"] for r in source["results"]))
        for method in methods:
            rows = [r for r in source["results"] if r["process"] == process and r["method"] == method]
            label = f"discovery/{phase}" if method == "discover" else method
            row = {"process": process, "method": label, "draws": len(rows)}
            for metric in ("coverage", "width", "interval_score", "coverage_low", "coverage_high", "elapsed"):
                row[metric], row[f"{metric}_se"] = average_se([r[metric] for r in rows])
            row["empty_sets"] = sum(r["empty_sets"] for r in rows)
            summary.append(row)
summary_frame = pl.DataFrame(summary)
competitors = ["catboost_split", "catboost_cqr", "lightgbm_split", "lightgbm_cqr", "sklearn_cqr"]
leaders = {}
for process in processes:
    eligible = [r for r in summary if r["process"] == process and r["method"] in competitors
                and abs(r["coverage"] - .9) <= .02]
    leaders[process] = min(eligible, key=lambda row: row["width"])["method"]


In [3]:
headline = summary_frame.select("process", "method", "draws", "coverage", "width", "width_se", "interval_score", "interval_score_se")
table = GT(headline, groupname_col="process", rowname_col="method").tab_header(
    title="Prediction intervals across both information tracks",
    subtitle="Nominal coverage 0.90; response-unit width and interval score: lower is better").fmt_number(columns=["width", "width_se", "interval_score_se"], decimals=3)
for index, row in enumerate(headline.iter_rows(named=True)):
    if row["method"] == leaders[row["process"]]:
        table = table.fmt(lambda value: f"<strong>{value:.3f}</strong>", columns="width", rows=[index])
display(table.fmt_number(columns=["coverage", "interval_score"], decimals=3)
    .tab_spanner("Width: mean and MC SE", ["width", "width_se"])
    .tab_spanner("Score: mean and MC SE", ["interval_score", "interval_score_se"])
    .tab_source_note("Known: true formula/family; discovery: only data and response. Competitors receive data only. "
        "Bold marks the retrospective lowest-width competitor within 0.02 of nominal coverage, not an overall ranking across information tracks. "
        "20 independent draws per row. Monte Carlo SE and paired differences follow below. Interval score is the scaled two-quantile score defined in the scoring addendum."))


GT(_tbl_data=┌────────────┬──────────────┬───────┬──────────┬──────────┬──────────┬──────────────┬──────────────┐
│ process    ┆ method       ┆ draws ┆ coverage ┆ width    ┆ width_se ┆ interval_sco ┆ interval_sco │
│            ┆              ┆       ┆          ┆          ┆          ┆ re           ┆ re_se        │
╞════════════╪══════════════╪═══════╪══════════╪══════════╪══════════╪══════════════╪══════════════╡
│ absolute   ┆ discovery/ma ┆ 20    ┆ 0.9002   ┆ 3.849519 ┆ 0.034155 ┆ 5.212726     ┆ 0.044819     │
│            ┆ in           ┆       ┆          ┆          ┆          ┆              ┆              │
│ absolute   ┆ known        ┆ 20    ┆ 0.900175 ┆ 3.673226 ┆ 0.034892 ┆ 4.6762       ┆ 0.021513     │
│ absolute   ┆ catboost_spl ┆ 20    ┆ 0.897175 ┆ 4.1109   ┆ 0.049197 ┆ 5.673169     ┆ 0.039435     │
│            ┆ it           ┆       ┆          ┆          ┆          ┆              ┆              │
│ absolute   ┆ catboost_cqr ┆ 20    ┆ 0.897175 ┆ 3.905329 ┆ 0.037666 ┆ 5.541047     ┆ 0.067708     │
│ absolute   ┆ lightgbm_spl ┆ 20    ┆ 0.897275 ┆ 4.190041 ┆ 0.055502 ┆ 5.718108     ┆ 0.032304     │
│            ┆ it           ┆       ┆          ┆          ┆          ┆              ┆              │
│ …          ┆ …            ┆ …     ┆ …        ┆ …        ┆ …        ┆ …            ┆ …            │
│ heavy_tail ┆ catboost_cqr ┆ 20    ┆ 0.903025 ┆ 3.449959 ┆ 0.069717 ┆ 5.096816     ┆ 0.074227     │
│ heavy_tail ┆ lightgbm_spl ┆ 20    ┆ 0.901475 ┆ 3.168003 ┆ 0.05436  ┆ 4.82895      ┆ 0.061639     │
│            ┆ it           ┆       ┆          ┆          ┆          ┆              ┆              │
│ heavy_tail ┆ lightgbm_cqr ┆ 20    ┆ 0.89805  ┆ 3.18593  ┆ 0.059798 ┆ 4.837458     ┆ 0.072862     │
│ heavy_tail ┆ sklearn_cqr  ┆ 20    ┆ 0.900575 ┆ 3.237755 ┆ 0.056902 ┆ 4.866143     ┆ 0.063205     │
│ heavy_tail ┆ discovery/cu ┆ 20    ┆ 0.900625 ┆ 2.799407 ┆ 0.042388 ┆ 4.471338     ┆ 0.056855     │
│            ┆ rve          ┆       ┆          ┆          ┆          ┆              ┆              │
└────────────┴──────────────┴───────┴──────────┴──────────┴──────────┴──────────────┴──────────────┘, _body=<great_tables._gt_data.Body object at 0x10bc1bc80>, _boxhead=Boxhead([ColInfo(var='process', type=<ColInfoTypeEnum.row_group: 3>, column_label='process', column_align='left', column_width=None, stub_level=None), ColInfo(var='method', type=<ColInfoTypeEnum.stub: 2>, column_label='method', column_align='left', column_width=None, stub_level=0), ColInfo(var='draws', type=<ColInfoTypeEnum.default: 1>, column_label='draws', column_align='right', column_width=None, stub_level=None), ColInfo(var='coverage', type=<ColInfoTypeEnum.default: 1>, column_label='coverage', column_align='right', column_width=None, stub_level=None), ColInfo(var='width', type=<ColInfoTypeEnum.default: 1>, column_label='width', column_align='right', column_width=None, stub_level=None), ColInfo(var='width_se', type=<ColInfoTypeEnum.default: 1>, column_label='width_se', column_align='right', column_width=None, stub_level=None), ColInfo(var='interval_score', type=<ColInfoTypeEnum.default: 1>, column_label='interval_score', column_align='right', column_width=None, stub_level=None), ColInfo(var='interval_score_se', type=<ColInfoTypeEnum.default: 1>, column_label='interval_score_se', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10c00e030>, _spanners=Spanners([SpannerInfo(spanner_id='Width: mean and MC SE', spanner_level=0, spanner_label='Width: mean and MC SE', spanner_units=None, spanner_pattern=None, vars=['width', 'width_se'], built=None), SpannerInfo(spanner_id='Score: mean and MC SE', spanner_level=0, spanner_label='Score: mean and MC SE', spanner_units=None, spanner_pattern=None, vars=['interval_score', 'interval_score_se'], built=None)]), _heading=Heading(title='Prediction intervals across both information tracks', subtitle='Nominal coverage 0.90; response-unit width and interval score: lower is bette

## Study design

Each draw has 1,000 development rows and 2,000 independent test rows. Predictors are independent uniforms on `[-1, 1]`; the mean is `sin(2*pi*x1) + 0.5*x2`. The absolute, monotone and bell scales are respectively `0.3 + 1.5*abs(x1)`, `exp(0.7*x1)` and `0.3 + 1.5*exp(-8*x1*x1)`. The constant-scale control uses standard normal errors. The heavy-tail control uses a unit-variance Student t distribution with three degrees of freedom and violates the diagnostic's Gaussian-errors assumption.

The [protocol](nonmonotone_spread/PROTOCOL.md) sets the design and seeds. Original seeds 0–5 reproduce the report; confirmation seeds 562100–562119 are separate independent draws, with test seed offset 50000. The settings do not change with the test scores.

Discovery receives only the development data and response name, with its default mean/interaction search, stopping, refit and calibration. The known-specification estimator receives `bbs(x1, df=4) + bols(x2)` and Gaussian family; on the varying-scale processes it also receives a smooth scale term for `x1`. Its stopping budget is fixed at 100 updates. Competitors use the same 500 fit and 500 calibration rows on each draw. Discovery uses all development rows during selection, before its internal refit and calibration path, so the two tracks also differ in data use. The calibration responses have already influenced discovery, so the internal split does not give a whole-procedure coverage guarantee.

CatBoost split/CQR use 500 depth-six trees at learning rate 0.05. LightGBM split/CQR use 300 trees with 15 leaves at rate 0.05. Sklearn CQR uses 300 depth-three trees at rate 0.05. Settings are untuned; every method uses one thread. Elapsed times measure whole workflows, not implementation speed ratios. Full warnings, source/package/build identities, input and prediction digests, chosen formulas and discovery-stage decisions accompany every draw.

## Spread nomination

The diagnostic fits a straight line to log-squared residuals. Opposite slopes on the two sides of a symmetric scale pattern cancel. The nomination measures the held-out squared-error reduction from eight quantile-bin means of those residuals, averaged over five fixed half-splits. Rust computes the bins, held-half scores and averaging through existing core APIs. These overlapping splits stabilize a nomination; they do not supply independent significance tests.

The following public-API example reproduces the absolute-scale process. Discovery receives neither its mean nor its scale formula.

In [4]:
import mboostrs as mb
rng = np.random.default_rng(0)
x1, x2 = rng.uniform(-1, 1, (2, 1000))
y = np.sin(2*np.pi*x1) + .5*x2 + (.3 + 1.5*np.abs(x1))*rng.normal(size=1000)
training = pl.DataFrame({"x1": x1, "x2": x2, "y": y})

# This preliminary mean fit exposes the measurement used for nomination.
mean_fit = mb.gamboost("y ~ bbs(x1, df=4) + bbs(x2, df=4)", training,
                      control=mb.boost_control(200))
diagnostic = mb.heteroscedasticity(mean_fit, columns=["x1"])["by_column"]["x1"]
display(GT(pl.DataFrame([diagnostic])).tab_header(title="Slope and curve evidence for x1",
    subtitle="Signed slope can cancel; positive held-out curve gain nominates a scale term")
    .fmt_number(columns=["slope", "se", "t", "spread_ratio", "curve_removed"], decimals=3)
    .tab_source_note("The slope t is the existing normal-errors diagnostic. Curve gain has squared log-squared-residual units; it is not a p-value."))


GT(_tbl_data=┌──────────┬──────────┬──────────┬───────────┬──────────────┬───────────────┐
│ slope    ┆ se       ┆ t        ┆ transform ┆ spread_ratio ┆ curve_removed │
╞══════════╪══════════╪══════════╪═══════════╪══════════════╪═══════════════╡
│ 0.072675 ┆ 0.123422 ┆ 0.588831 ┆ null      ┆ 1.058076     ┆ 0.490128      │
└──────────┴──────────┴──────────┴───────────┴──────────────┴───────────────┘, _body=<great_tables._gt_data.Body object at 0x10cd23ce0>, _boxhead=Boxhead([ColInfo(var='slope', type=<ColInfoTypeEnum.default: 1>, column_label='slope', column_align='right', column_width=None, stub_level=None), ColInfo(var='se', type=<ColInfoTypeEnum.default: 1>, column_label='se', column_align='right', column_width=None, stub_level=None), ColInfo(var='t', type=<ColInfoTypeEnum.default: 1>, column_label='t', column_align='right', column_width=None, stub_level=None), ColInfo(var='transform', type=<ColInfoTypeEnum.default: 1>, column_label='transform', column_align='center', column_width=None, stub_level=None), ColInfo(var='spread_ratio', type=<ColInfoTypeEnum.default: 1>, column_label='spread_ratio', column_align='right', column_width=None, stub_level=None), ColInfo(var='curve_removed', type=<ColInfoTypeEnum.default: 1>, column_label='curve_removed', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10b4b6a80>, _spanners=Spanners([]), _heading=Heading(title='Slope and curve evidence for x1', subtitle='Signed slope can cancel; positive held-out curve gain nominates a scale term', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x110c0ffb0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x110cce030>, _source_notes=['The slope t is the existing normal-errors diagnostic. Curve gain has squared log-squared-residual units; it is not a p-value.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x110c69a60>, _formats=[<great_tables._gt_data.FormatInfo object at 0x110cfb8c0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=Tr

In [5]:
prediction_grid = pl.DataFrame({"x1": [-.9, -.45, 0., .45, .9], "x2": [0.] * 5})
result = mb.discover(training, "y", newdata=prediction_grid, seed=0, level=.9)
assert result.spread_formula == "bbs(x1, df=4)"
nomination = next(row for row in result.log if "nomination_scores" in row)
selected = next(row for row in result.log if row["stage"] == "spread" and row.get("adopted"))
adoption_change, adoption_se = selected["change"], selected["se"]
display(GT(prediction_grid.hstack(result.predictions.drop("row"))).tab_header(
    title="Discovered mean, scale and prediction band",
    subtitle="Response units; x2 held at zero; nominal coverage 90%")
    .fmt_number(columns=["x1", "x2", "mean", "sigma", "lower", "upper"], decimals=3)
    .tab_source_note("Sigma is the final model's predicted standard deviation; lower/upper come from the calibrated prediction interval."))


GT(_tbl_data=┌───────┬─────┬───────────┬──────────┬───────────┬──────────┐
│ x1    ┆ x2  ┆ mean      ┆ sigma    ┆ lower     ┆ upper    │
╞═══════╪═════╪═══════════╪══════════╪═══════════╪══════════╡
│ -0.9  ┆ 0.0 ┆ 0.916828  ┆ 1.511608 ┆ -1.658671 ┆ 3.492327 │
│ -0.45 ┆ 0.0 ┆ -0.161674 ┆ 0.923354 ┆ -1.76958  ┆ 1.446233 │
│ 0.0   ┆ 0.0 ┆ 0.105128  ┆ 0.37983  ┆ -0.608763 ┆ 0.819018 │
│ 0.45  ┆ 0.0 ┆ 0.419353  ┆ 0.914563 ┆ -1.174095 ┆ 2.012801 │
│ 0.9   ┆ 0.0 ┆ -1.004661 ┆ 1.806202 ┆ -4.064723 ┆ 2.055402 │
└───────┴─────┴───────────┴──────────┴───────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x115d273b0>, _boxhead=Boxhead([ColInfo(var='x1', type=<ColInfoTypeEnum.default: 1>, column_label='x1', column_align='right', column_width=None, stub_level=None), ColInfo(var='x2', type=<ColInfoTypeEnum.default: 1>, column_label='x2', column_align='right', column_width=None, stub_level=None), ColInfo(var='mean', type=<ColInfoTypeEnum.default: 1>, column_label='mean', column_align='right', column_width=None, stub_level=None), ColInfo(var='sigma', type=<ColInfoTypeEnum.default: 1>, column_label='sigma', column_align='right', column_width=None, stub_level=None), ColInfo(var='lower', type=<ColInfoTypeEnum.default: 1>, column_label='lower', column_align='right', column_width=None, stub_level=None), ColInfo(var='upper', type=<ColInfoTypeEnum.default: 1>, column_label='upper', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x116390110>, _spanners=Spanners([]), _heading=Heading(title='Discovered mean, scale and prediction band', subtitle='Response units; x2 held at zero; nominal coverage 90%', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1157437a0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x111b79700>, _source_notes=["Sigma is the final model's predicted standard deviation; lower/upper come from the calibrated prediction interval."], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x116453650>, _formats=[<great_tables._gt_data.FormatInfo object at 0x111cc6570>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsI

A nominated column still has to improve the held-out Gaussian location-scale score (the held-out log score of a normal model whose mean and scale each have an additive predictor; lower is better) by at least two change standard errors. In the displayed draw the change is {eval}`adoption_change`, with change SE {eval}`adoption_se`. The candidate restrictions and the cost limit still apply. The preliminary mean residuals come from the fitted development data; they are not cross-fitted residuals (residuals from models fitted on other folds), and their apparent structure alone does not identify a true variance mechanism.

In [6]:
# Pair exactly the same input/test draw for each version and reference.
lookup = {(r["process"], r["seed"], r["method"], r["phase"]): r for r in records}
gaps = []
for process in processes:
    best = leaders[process]
    for phase, method in [("final_baseline", "discover"), ("final_fixed", "discover"), ("final_baseline", "known")]:
        paired = []
        for seed in range(562100, 562120):
            row = lookup[process, seed, method, phase]
            peer = lookup[process, seed, best, "final_baseline"]
            assert row["input_digest"] == peer["input_digest"]
            paired.append(100*(row["width"]/peer["width"] - 1))
        gap, se = average_se(paired)
        gaps.append({"process": process, "track": "known" if method == "known" else phase,
                     "retrospective_competitor": best, "width_gap_pct": gap, "MC_SE_points": se})
gap_frame = pl.DataFrame(gaps)
display(GT(gap_frame, groupname_col="process").tab_header(title="Width gaps by information track",
    subtitle="Percent relative to retrospective eligible competitor; negative favors mboostrs")
    .fmt_number(columns=["width_gap_pct", "MC_SE_points"], decimals=2)
    .tab_source_note("Winner selected from test-set mean widths among competitors within 0.02 of nominal coverage. MC SE is paired across 20 independent draws; it does not account for winner selection. Known and discovery have different information."))


GT(_tbl_data=┌────────────┬────────────────┬──────────────────────────┬───────────────┬──────────────┐
│ process    ┆ track          ┆ retrospective_competitor ┆ width_gap_pct ┆ MC_SE_points │
╞════════════╪════════════════╪══════════════════════════╪═══════════════╪══════════════╡
│ absolute   ┆ final_baseline ┆ lightgbm_cqr             ┆ 4.971098      ┆ 1.277641     │
│ absolute   ┆ final_fixed    ┆ lightgbm_cqr             ┆ 0.015654      ┆ 1.165214     │
│ absolute   ┆ known          ┆ lightgbm_cqr             ┆ 0.13667       ┆ 1.157367     │
│ monotone   ┆ final_baseline ┆ lightgbm_cqr             ┆ 1.818194      ┆ 0.945519     │
│ monotone   ┆ final_fixed    ┆ lightgbm_cqr             ┆ 1.818194      ┆ 0.945519     │
│ …          ┆ …              ┆ …                        ┆ …             ┆ …            │
│ constant   ┆ final_fixed    ┆ catboost_split           ┆ -3.960721     ┆ 1.220847     │
│ constant   ┆ known          ┆ catboost_split           ┆ 0.307511      ┆ 1.478358     │
│ heavy_tail ┆ final_baseline ┆ catboost_split           ┆ -7.265647     ┆ 1.772444     │
│ heavy_tail ┆ final_fixed    ┆ catboost_split           ┆ -7.265647     ┆ 1.772444     │
│ heavy_tail ┆ known          ┆ catboost_split           ┆ -1.257698     ┆ 1.82042      │
└────────────┴────────────────┴──────────────────────────┴───────────────┴──────────────┘, _body=<great_tables._gt_data.Body object at 0x11622b440>, _boxhead=Boxhead([ColInfo(var='process', type=<ColInfoTypeEnum.row_group: 3>, column_label='process', column_align='left', column_width=None, stub_level=None), ColInfo(var='track', type=<ColInfoTypeEnum.default: 1>, column_label='track', column_align='left', column_width=None, stub_level=None), ColInfo(var='retrospective_competitor', type=<ColInfoTypeEnum.default: 1>, column_label='retrospective_competitor', column_align='left', column_width=None, stub_level=None), ColInfo(var='width_gap_pct', type=<ColInfoTypeEnum.default: 1>, column_label='width_gap_pct', column_align='right', column_width=None, stub_level=None), ColInfo(var='MC_SE_points', type=<ColInfoTypeEnum.default: 1>, column_label='MC_SE_points', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10c861c10>, _spanners=Spanners([]), _heading=Heading(title='Width gaps by information track', subtitle='Percent relative to retrospective eligible competitor; negative favors mboostrs', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x115b7cf20>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x115b7cd10>, _source_notes=['Winner selected from test-set mean widths among competitors within 0.02 of nominal coverage. MC SE is paired across 20 independent draws; it does not account for winner selection. Known and discovery have different information.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x115b7cd40>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10cd9cbc0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'san

In [7]:
enhancements = []
for process in processes:
    pairs = [(lookup[process, seed, "discover", "final_baseline"],
              lookup[process, seed, "discover", "final_fixed"]) for seed in range(562100, 562120)]
    assert all(a["input_digest"] == b["input_digest"] for a, b in pairs)
    row = {"process": process}
    for name, values in {
        "width_change_pct": [100*(b["width"]/a["width"]-1) for a, b in pairs],
        "coverage_change_points": [100*(b["coverage"]-a["coverage"]) for a, b in pairs],
        "score_change": [b["interval_score"]-a["interval_score"] for a, b in pairs]}.items():
        row[name], row[name+"_se"] = average_se(values)
    row["main_adoptions"] = sum(a["details"]["spread_formula"] is not None for a, b in pairs)
    row["curve_adoptions"] = sum(b["details"]["spread_formula"] is not None for a, b in pairs)
    enhancements.append(row)
enhancement_frame = pl.DataFrame(enhancements)
display(GT(enhancement_frame).tab_header(title="Paired effect of the nomination change",
    subtitle="Curve screen minus main; lower width/score is better; coverage is a percentage-point change")
    .fmt_number(columns=[name for name in enhancement_frame.columns if "change" in name], decimals=3)
    .tab_source_note("All 20 paired draws retained, including unchanged draws. SE columns are Monte Carlo standard errors across independent draws, not CV-fold uncertainty. Adoption counts are out of 20."))


GT(_tbl_data=┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ process   ┆ width_cha ┆ width_cha ┆ coverage_ ┆ … ┆ score_cha ┆ score_cha ┆ main_adop ┆ curve_ad │
│           ┆ nge_pct   ┆ nge_pct_s ┆ change_po ┆   ┆ nge       ┆ nge_se    ┆ tions     ┆ options  │
│           ┆           ┆ e         ┆ ints      ┆   ┆           ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ absolute  ┆ -4.629945 ┆ 0.888776  ┆ -0.1125   ┆ … ┆ -0.535497 ┆ 0.035604  ┆ 1         ┆ 20       │
│ monotone  ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 20        ┆ 20       │
│ bell      ┆ -8.184239 ┆ 1.87831   ┆ 0.3975    ┆ … ┆ -1.096571 ┆ 0.085619  ┆ 2         ┆ 20       │
│ constant  ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0         ┆ 0        │
│ heavy_tai ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0         ┆ 0        │
│ l         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10cd23a10>, _boxhead=Boxhead([ColInfo(var='process', type=<ColInfoTypeEnum.default: 1>, column_label='process', column_align='left', column_width=None, stub_level=None), ColInfo(var='width_change_pct', type=<ColInfoTypeEnum.default: 1>, column_label='width_change_pct', column_align='right', column_width=None, stub_level=None), ColInfo(var='width_change_pct_se', type=<ColInfoTypeEnum.default: 1>, column_label='width_change_pct_se', column_align='right', column_width=None, stub_level=None), ColInfo(var='coverage_change_points', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_change_points', column_align='right', column_width=None, stub_level=None), ColInfo(var='coverage_change_points_se', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_change_points_se', column_align='right', column_width=None, stub_level=None), ColInfo(var='score_change', type=<ColInfoTypeEnum.default: 1>, column_label='score_change', column_align='right', column_width=None, stub_level=None), ColInfo(var='score_change_se', type=<ColInfoTypeEnum.default: 1>, column_label='score_change_se', column_align='right', column_width=None, stub_level=None), ColInfo(var='main_adoptions', type=<ColInfoTypeEnum.default: 1>, column_label='main_adoptions', column_align='right', column_width=None, stub_level=None), ColInfo(var='curve_adoptions', type=<ColInfoTypeEnum.default: 1>, column_label='curve_adoptions', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x111b57c50>, _spanners=Spanners([]), _heading=Heading(title='Paired effect of the nomination change', subtitle='Curve screen minus main; lower width/score is better; coverage is a percentage-point change', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1161eef60>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x111b54860>, _source_notes=['All 20 paired draws retained, including unchanged draws. SE columns are Monte Carlo standard errors across independent draws, not CV-fold uncertainty. Adoption counts are out of 20.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x111b54800>, _formats=[<great_tables._gt_data.FormatInfo object at 0x111b57bf0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='a

In [8]:
bell_gap = next(r for r in gaps if r["process"] == "bell" and r["track"] == "final_fixed")
monotone_gap = next(r for r in gaps if r["process"] == "monotone" and r["track"] == "final_fixed")
absolute_change = next(r for r in enhancements if r["process"] == "absolute")
bell_change = next(r for r in enhancements if r["process"] == "bell")
absolute_width_change = absolute_change["width_change_pct"]
absolute_width_se = absolute_change["width_change_pct_se"]
bell_width_change = bell_change["width_change_pct"]
bell_width_se = bell_change["width_change_pct_se"]
conditional = summary_frame.select("process", "method", "coverage", "coverage_se", "coverage_low", "coverage_high", "empty_sets")
display(GT(conditional, groupname_col="process").tab_header(title="Marginal and conditional coverage",
    subtitle="Shares of test responses covered; target 0.90; empty-set count over 40,000 predictions")
    .fmt_number(columns=["coverage", "coverage_se", "coverage_low", "coverage_high"], decimals=3)
    .tab_source_note("Low/high groups are the lower/upper thirds of true test scale, available only to the evaluator. Constant-scale processes put every row in both groups. Empty CQR sets have zero width and no coverage; their two-quantile score remains recorded."))


GT(_tbl_data=┌────────────┬────────────────┬──────────┬─────────────┬──────────────┬───────────────┬────────────┐
│ process    ┆ method         ┆ coverage ┆ coverage_se ┆ coverage_low ┆ coverage_high ┆ empty_sets │
╞════════════╪════════════════╪══════════╪═════════════╪══════════════╪═══════════════╪════════════╡
│ absolute   ┆ discovery/main ┆ 0.9002   ┆ 0.003366    ┆ 0.993178     ┆ 0.783508      ┆ 0          │
│ absolute   ┆ known          ┆ 0.900175 ┆ 0.003426    ┆ 0.914993     ┆ 0.893028      ┆ 0          │
│ absolute   ┆ catboost_split ┆ 0.897175 ┆ 0.002863    ┆ 0.997001     ┆ 0.765367      ┆ 0          │
│ absolute   ┆ catboost_cqr   ┆ 0.897175 ┆ 0.002515    ┆ 0.966417     ┆ 0.802924      ┆ 53         │
│ absolute   ┆ lightgbm_split ┆ 0.897275 ┆ 0.003736    ┆ 0.995802     ┆ 0.774738      ┆ 0          │
│ …          ┆ …              ┆ …        ┆ …           ┆ …            ┆ …             ┆ …          │
│ heavy_tail ┆ catboost_cqr   ┆ 0.903025 ┆ 0.003116    ┆ 0.903025     ┆ 0.903025      ┆ 69         │
│ heavy_tail ┆ lightgbm_split ┆ 0.901475 ┆ 0.003434    ┆ 0.901475     ┆ 0.901475      ┆ 0          │
│ heavy_tail ┆ lightgbm_cqr   ┆ 0.89805  ┆ 0.003084    ┆ 0.89805      ┆ 0.89805       ┆ 0          │
│ heavy_tail ┆ sklearn_cqr    ┆ 0.900575 ┆ 0.003094    ┆ 0.900575     ┆ 0.900575      ┆ 5          │
│ heavy_tail ┆ discovery/curv ┆ 0.900625 ┆ 0.003798    ┆ 0.900625     ┆ 0.900625      ┆ 0          │
│            ┆ e              ┆          ┆             ┆              ┆               ┆            │
└────────────┴────────────────┴──────────┴─────────────┴──────────────┴───────────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x115d273b0>, _boxhead=Boxhead([ColInfo(var='process', type=<ColInfoTypeEnum.row_group: 3>, column_label='process', column_align='left', column_width=None, stub_level=None), ColInfo(var='method', type=<ColInfoTypeEnum.default: 1>, column_label='method', column_align='left', column_width=None, stub_level=None), ColInfo(var='coverage', type=<ColInfoTypeEnum.default: 1>, column_label='coverage', column_align='right', column_width=None, stub_level=None), ColInfo(var='coverage_se', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_se', column_align='right', column_width=None, stub_level=None), ColInfo(var='coverage_low', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_low', column_align='right', column_width=None, stub_level=None), ColInfo(var='coverage_high', type=<ColInfoTypeEnum.default: 1>, column_label='coverage_high', column_align='right', column_width=None, stub_level=None), ColInfo(var='empty_sets', type=<ColInfoTypeEnum.default: 1>, column_label='empty_sets', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10c36adb0>, _spanners=Spanners([]), _heading=Heading(title='Marginal and conditional coverage', subtitle='Shares of test responses covered; target 0.90; empty-set count over 40,000 predictions', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x115a9aa80>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10cd46b40>, _source_notes=['Low/high groups are the lower/upper thirds of true test scale, available only to the evaluator. Constant-scale processes put every row in both groups. Empty CQR sets have zero width and no coverage; their two-quantile score remains recorded.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x115b1e1b0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x115af39b0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scs

In [9]:
cost = summary_frame.filter(pl.col("method").str.starts_with("discovery/")).select(
    "process", "method", "elapsed", "elapsed_se")
display(GT(cost, groupname_col="process").tab_header(title="Recorded discovery workflow cost",
    subtitle="Elapsed seconds per draw; mean and Monte Carlo SE across 20 draws")
    .fmt_number(columns=["elapsed", "elapsed_se"], decimals=3)
    .tab_source_note("Includes fitting, selection, refit and prediction. Runs share the host with the paired baseline/control process; this is descriptive cost, not a controlled speed comparison. Existing cost admission remains in force and is covered by the zero-budget regression test."))

GT(_tbl_data=┌────────────┬─────────────────┬──────────┬────────────┐
│ process    ┆ method          ┆ elapsed  ┆ elapsed_se │
╞════════════╪═════════════════╪══════════╪════════════╡
│ absolute   ┆ discovery/main  ┆ 0.114397 ┆ 0.041707   │
│ absolute   ┆ discovery/curve ┆ 0.783803 ┆ 0.027781   │
│ monotone   ┆ discovery/main  ┆ 0.791305 ┆ 0.050822   │
│ monotone   ┆ discovery/curve ┆ 0.777321 ┆ 0.050025   │
│ bell       ┆ discovery/main  ┆ 0.120484 ┆ 0.028499   │
│ bell       ┆ discovery/curve ┆ 0.475059 ┆ 0.030567   │
│ constant   ┆ discovery/main  ┆ 0.067395 ┆ 0.003345   │
│ constant   ┆ discovery/curve ┆ 0.068162 ┆ 0.004383   │
│ heavy_tail ┆ discovery/main  ┆ 0.070931 ┆ 0.005117   │
│ heavy_tail ┆ discovery/curve ┆ 0.071141 ┆ 0.005253   │
└────────────┴─────────────────┴──────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x1155efd70>, _boxhead=Boxhead([ColInfo(var='process', type=<ColInfoTypeEnum.row_group: 3>, column_label='process', column_align='left', column_width=None, stub_level=None), ColInfo(var='method', type=<ColInfoTypeEnum.default: 1>, column_label='method', column_align='left', column_width=None, stub_level=None), ColInfo(var='elapsed', type=<ColInfoTypeEnum.default: 1>, column_label='elapsed', column_align='right', column_width=None, stub_level=None), ColInfo(var='elapsed_se', type=<ColInfoTypeEnum.default: 1>, column_label='elapsed_se', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x1117eba70>, _spanners=Spanners([]), _heading=Heading(title='Recorded discovery workflow cost', subtitle='Elapsed seconds per draw; mean and Monte Carlo SE across 20 draws', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x111e03980>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10cd23950>, _source_notes=['Includes fitting, selection, refit and prediction. Runs share the host with the paired baseline/control process; this is descriptive cost, not a controlled speed comparison. Existing cost admission remains in force and is covered by the zero-budget regression test.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10cd23380>, _formats=[<great_tables._gt_data.FormatInfo object at 0x115af39b0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', t

## Results and remaining gaps

On the absolute-scale process, paired mean width changes by {eval}`absolute_width_change` percent (Monte Carlo SE {eval}`absolute_width_se` percentage points). The bell process changes by {eval}`bell_width_change` percent (SE {eval}`bell_width_se` points). Marginal and conditional coverage above distinguish narrower valid intervals from simple undercoverage. The screen selects the varying-scale driver on every fresh absolute and bell draw; the monotone control result is unchanged from the baseline. Neither smooth-mean constant-scale control gains a selected scale model in this batch. These finite samples do not prove a population false-selection rate.

The remaining width gaps to retrospective LightGBM CQR are {eval}`monotone_gap["width_gap_pct"]` percent (paired SE {eval}`monotone_gap["MC_SE_points"]` points) for monotone scale and {eval}`bell_gap["width_gap_pct"]` percent (SE {eval}`bell_gap["MC_SE_points"]` points) for bell scale. These remaining gaps are limitations of the method. These point estimates exceed the 1% width target despite selecting the varying-scale driver. Untuned competitors, fixed known-specification stopping and different data use limit that comparison. The exact oracle is a secondary bound, not a competing method. The broader discovery interval question is outside this study.

A separate sharp-interaction, constant-noise counterexample is reported here. The 200-step mean screening fit can leave approximation error that the held-out distributional score assigns to sigma. Its 2,000-row mean MSE changes from 0.573 to 0.515, so it does not show a mean-error regression, but its adopted scale driver is not a true noise mechanism. The smooth-mean null controls here do not cover that failure. No mean-adequacy policy is changed in this study.

## Scoring definitions

Width is `max(upper-lower, 0)`. An empty prediction set, which occurs when the lower bound lies above the upper bound, counts as not covering the response.

## Reproduction

Reproduction commands, run from the repository root, are `PYTHONPATH=<unchanged-main>/python:. .venv/bin/python -m benches.study_nonmonotone_spread --phase final_baseline` and `PYTHONPATH=python:. .venv/bin/python -m benches.study_nonmonotone_spread --phase final_fixed`. The runner refuses to overwrite evidence, so a new run needs a fresh checkout or output directory. The baseline and the screen must use the same release extension, built from the Rust and Cargo sources. The screen changes no native numerical source.

Within this study, both known-specification and discovery support **D9 score 3**: coverage is approximately nominal on these standard processes, with independent-draw uncertainty reported. The varying-width results demonstrate the heteroscedasticity part of score 4, but small-sample and extrapolation adaptation are unmeasured, so this study does not establish full score 4. D1a, D1b, D2–D8, D10 and D11 remain unmeasured here. This is an empirical assessment, subject to the discovery-selection limitation under Study design; it does not establish a formal coverage guarantee. No aggregate grade or scorecard file is changed.
